# 02 — QLoRA training (Colab T4)

Fine-tune `Qwen/Qwen2.5-3B` (base, not Instruct) so it continues a spec sheet with an asking price in EGP.

**This run**
- 8,000 lite training ads, one epoch
- 1,000 validation ads, scored only (they do not update the adapter)
- The 1,000 test cars are not loaded
- Loss is only on the completion, the digits after `Price is EGP`
- The saved file is a LoRA adapter, not a new copy of Qwen

Run this on a Colab **T4** (Runtime → Change runtime type → T4 GPU). A laptop CPU will not finish this. Do not push the adapter to the Hub unless you explicitly ask to.

## 1. Setup

On Colab this cell clones the repo and installs the train extra (`torch`, `transformers`, `trl`, `peft`, `bitsandbytes`). `data/` is gitignored, so a fresh clone has no parquet files. A later cell rebuilds them from the Hub with the same prep code.

If the install cell just ran for the first time, use **Runtime → Restart session**, then run every cell from the next one down. A new `torch` does not become importable in the current process.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
REPO_URL = "https://github.com/MohamedAlaa2180/egyptian-car-pricer.git"


def find_root() -> Path | None:
    here = Path.cwd().resolve()
    candidates = [here, here.parent, Path("/content/egyptian-car-pricer")]
    for candidate in candidates:
        if (candidate / "pricer" / "prep.py").exists():
            return candidate
    return None


if IN_COLAB and find_root() is None:
    dest = Path("/content/egyptian-car-pricer")
    subprocess.check_call(["git", "clone", REPO_URL, str(dest)])
    os.chdir(dest)

ROOT = find_root()
if ROOT is None:
    raise SystemExit(
        "Repo root not found. On Colab, clone the project under /content. "
        "Locally, open this notebook from the repo."
    )
os.chdir(ROOT)
print("Repo:", ROOT)

if IN_COLAB:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"]
    )
    print("Installed .[train]. Restart the runtime if this was the first install, then rerun from the next cell.")
else:
    print("Local runtime. Install with: pip install -e \".[train]\" on the GPU machine.")

## 2. GPU check and logins

QLoRA needs a CUDA GPU. The cell stops here on a CPU so a multi-hour CPU run never starts.

`Qwen/Qwen2.5-3B` is a public base model. A Hugging Face token is only needed if the Hub rate-limits the download.

Training logs go to the Weights & Biases project `mohamedalaasalem1/egyptian-car-pricer`. Put the key in `.env` as `WANDB_API_KEY`, or on Colab add a secret with that name. Do not commit the key. The adapter weights stay on disk; they are not uploaded as a W&B artifact.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pricer" / "prep.py").exists() and (ROOT.parent / "pricer" / "prep.py").exists():
    ROOT = ROOT.parent.resolve()
    os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

try:
    import torch
except ImportError as exc:
    raise SystemExit(
        "torch is not installed. On Colab, run the setup cell and restart the runtime. "
        "On a CUDA machine: pip install -e \".[train]\""
    ) from exc
try:
    import wandb
except ImportError as exc:
    raise SystemExit(
        "wandb is not installed. Re-run the setup cell so pip install -e \".[train]\" picks it up, then restart the runtime."
    ) from exc
from dotenv import load_dotenv
from huggingface_hub import login

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU. Open this notebook on a Colab T4: "
        "Runtime → Change runtime type → T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
total_gb = getattr(props, "total_gb", None) or props.total_memory / 1024**3
print(f"GPU: {gpu_name}  ({total_gb:.1f} GB)")

def secret(name: str) -> str | None:
    value = os.getenv(name)
    if value:
        return value
    if "COLAB_RELEASE_TAG" in os.environ:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return None


load_dotenv(ROOT / ".env", override=True)
token = secret("HF_TOKEN")
if token:
    login(token, add_to_git_credential=False)
    print("Logged in to Hugging Face")
else:
    print("No HF_TOKEN. The public base model can still download.")

wandb_key = secret("WANDB_API_KEY")
if not wandb_key:
    raise SystemExit(
        "Set WANDB_API_KEY in .env, or as a Colab secret of that name. "
        "Create the key at https://wandb.ai/authorize"
    )
os.environ["WANDB_API_KEY"] = wandb_key
os.environ["WANDB_ENTITY"] = "mohamedalaasalem1"
os.environ["WANDB_PROJECT"] = "egyptian-car-pricer"
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_DIR"] = str(ROOT / "wandb")
wandb.login(key=wandb_key)
print("Logged in to Weights & Biases")

## 3. Load the lite train set and the validation set

`LITE = True` reads `data/train_lite.parquet` (8,000 ads). Set it to `False` later for the full 14,182-row train file. Validation stays 1,000 ads either way.

The test parquet is not read. If `data/` is missing (a fresh Colab clone), the cell reruns the prep filters and writes the splits, then still hands the trainer only lite and val.

In [ ]:
import pandas as pd
from datasets import Dataset

from pricer.items import PREFIX
from pricer.prep import (
    LITE_TRAIN_SIZE,
    SOURCE_DATASET,
    apply_hard_filters,
    apply_price_outliers,
    drop_exact_duplicates,
    lite_train,
    load_source,
    parse_frame,
    save_splits,
    split_frame,
)

LITE = True
DATA_DIR = ROOT / "data"


def rebuild_splits(data_dir):
    """Same filters as notebook 01. Writes test to disk and does not return it."""
    raw = load_source()
    parsed, n_fail = parse_frame(raw)
    kept, dropped_hard = apply_hard_filters(parsed)
    kept, n_dup = drop_exact_duplicates(kept)
    kept, dropped_out = apply_price_outliers(kept)
    train, val, test = split_frame(kept)
    lite = lite_train(train, n=LITE_TRAIN_SIZE)
    report = {
        "source_dataset": SOURCE_DATASET,
        "source_rows": int(len(raw)),
        "hub_splits": {k: int(v) for k, v in raw["hub_split"].value_counts().to_dict().items()},
        "hub_test_skipped": 4999,
        "hub_test_skip_reason": "completion is 0 on every row",
        "parse_failed": int(n_fail),
        "hard_drop": {k: int(v) for k, v in dropped_hard["drop_reason"].value_counts().to_dict().items()} if len(dropped_hard) else {},
        "exact_duplicates": int(n_dup),
        "price_outliers": int(len(dropped_out)),
        "kept": int(len(kept)),
        "gen_suspect_kept": int(kept["gen_suspect"].sum()),
        "splits": {
            "train": int(len(train)),
            "train_lite": int(len(lite)),
            "val": int(len(val)),
            "test": int(len(test)),
        },
    }
    save_splits(train, val, test, report, lite=lite, data_dir=data_dir)
    print(f"Rebuilt splits. Test rows written and left closed: {len(test):,}")
    return lite, val


lite_path = DATA_DIR / "train_lite.parquet"
full_path = DATA_DIR / "train.parquet"
val_path = DATA_DIR / "val.parquet"
train_path = lite_path if LITE else full_path

if train_path.exists() and val_path.exists():
    print("Loading", train_path.name, "and val.parquet")
    train_df = pd.read_parquet(train_path)
    val_df = pd.read_parquet(val_path)
else:
    print("Local parquet missing. Rebuilding from the Hub.")
    train_df, val_df = rebuild_splits(DATA_DIR)
    if not LITE:
        train_df = pd.read_parquet(full_path)

train_df = train_df[["prompt", "completion"]].copy()
val_df = val_df[["prompt", "completion"]].copy()
train_df["prompt"] = train_df["prompt"].astype(str)
train_df["completion"] = train_df["completion"].astype(str)
val_df["prompt"] = val_df["prompt"].astype(str)
val_df["completion"] = val_df["completion"].astype(str)

if not train_df["prompt"].str.contains(PREFIX, regex=False).all():
    raise SystemExit(f"A training prompt does not contain {PREFIX!r}")
if not train_df["completion"].str.fullmatch(r"\d+").all():
    raise SystemExit("A training completion is not an integer price")

print(f"train {len(train_df):,}   val {len(val_df):,}   lite={LITE}")
print("test.parquet was not read")
print("\n--- prompt ---")
print(train_df.iloc[0]["prompt"])
print("--- completion ---")
print(train_df.iloc[0]["completion"])

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)

## 4. QLoRA

The base weights load in 4-bit NF4 and stay frozen. LoRA adds two small matrices on each attention and MLP projection. With rank `r = 16` and `lora_alpha = 32`, the adapter is scaled by `alpha / r = 2`.

A T4 has no fast bfloat16, so the compute dtype is float16 there. On an A100 or L4 the cell switches to bfloat16.

Prompts in this dataset are about 200 characters. `max_length = 256` leaves room for the price tokens. Truncation keeps the start of the sequence, which would cut the price off the end, so the next cell refuses to train if any row is longer than 256 tokens.

In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

MODEL_ID = "Qwen/Qwen2.5-3B"
MAX_LENGTH = 256
USE_BF16 = torch.cuda.is_bf16_supported()
DTYPE_NAME = "bfloat16" if USE_BF16 else "float16"
COMPUTE_DTYPE = getattr(torch, DTYPE_NAME)
print("compute dtype:", DTYPE_NAME)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

lengths = [
    len(tokenizer(prompt + completion, add_special_tokens=False)["input_ids"])
    for prompt, completion in zip(train_df["prompt"], train_df["completion"])
]
longest = max(lengths)
print(f"longest train row: {longest} tokens (max_length {MAX_LENGTH})")
if longest > MAX_LENGTH:
    raise SystemExit(
        "A row is longer than max_length. Truncation would drop the price at the end."
    )

## 5. One epoch

Micro-batch size is 1 because the T4 also has to hold activations. Gradients accumulate for 16 steps, so each optimizer update sees 16 ads. On 8,000 ads that is 500 updates, which is one epoch.

Every 100 updates the validation loss is measured and a checkpoint is written. `load_best_model_at_end` puts the lowest-validation-loss adapter back into memory when training finishes. The test set is not an argument to the trainer.

`completion_only_loss=True` is the default for a dataset that has `prompt` and `completion` columns. It is set here so the mask is visible. Packing is off so each ad stays its own sequence.

Train loss and validation loss are sent to Weights & Biases every 10 and 100 steps. Open the run at [mohamedalaasalem1/egyptian-car-pricer](https://wandb.ai/mohamedalaasalem1/egyptian-car-pricer).

In [ ]:
import wandb
from trl import SFTConfig, SFTTrainer

RUN_NAME = "qwen2.5-3b-lite" if LITE else "qwen2.5-3b-full"
OUTPUT_DIR = ROOT / "adapters" / RUN_NAME

wandb.init(
    entity="mohamedalaasalem1",
    project="egyptian-car-pricer",
    name=RUN_NAME,
    config={
        "model": MODEL_ID,
        "lite": LITE,
        "train_rows": len(train_ds),
        "val_rows": len(val_ds),
        "epochs": 1,
        "learning_rate": 2e-4,
        "gradient_accumulation_steps": 16,
        "lora_r": 16,
        "lora_alpha": 32,
        "max_length": MAX_LENGTH,
    },
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=MAX_LENGTH,
    packing=False,
    completion_only_loss=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb",
    run_name=RUN_NAME,
    seed=42,
    model_init_kwargs={
        "dtype": DTYPE_NAME,
        "attn_implementation": "sdpa",
    },
)

trainer = SFTTrainer(
    model=MODEL_ID,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
    quantization_config=quantization_config,
)
trainer.model.print_trainable_parameters()

## 6. Train

The first number is the validation loss **before** any update. At that point the LoRA `B` matrices are still zeros, so this is the frozen base model scoring the price tokens.

Then one epoch runs. A second epoch is a separate decision: run it only if the validation loss is still falling at the last step. This notebook does not start that second pass.

In [ ]:
before = trainer.evaluate()
print(f"val loss before any update: {before['eval_loss']:.4f}")

trainer.train()
if trainer.state.best_metric is not None:
    print(f"best val loss: {trainer.state.best_metric:.4f}")
print("best checkpoint:", trainer.state.best_model_checkpoint)

## 7. Read the curves and save the adapter

Train loss and validation loss are both cross-entropy on the price tokens. They are not the EGP error.

- Both curves fall, then flatten: stop. This adapter is ready for notebook 03.
- Validation loss falls, then rises: the saved file is already the lowest point, because `load_best_model_at_end` reloaded it.
- Both curves stay flat at the starting loss: the adapter did not learn. Do not judge prices yet.

MAE on the 1,000 test cars comes after this, in `notebooks/03_eval.ipynb`. That file is still a stub.

In [ ]:
import matplotlib.pyplot as plt

history = pd.DataFrame(trainer.state.log_history)
train_log = history.dropna(subset=["loss"])
eval_log = history.dropna(subset=["eval_loss"])

fig, ax = plt.subplots(figsize=(8, 4))
if len(train_log):
    ax.plot(train_log["step"], train_log["loss"], label="train loss")
if len(eval_log):
    ax.plot(eval_log["step"], eval_log["eval_loss"], marker="o", label="val loss")
ax.set_xlabel("optimizer step")
ax.set_ylabel("loss on price tokens")
ax.set_title(RUN_NAME)
ax.legend()
fig.tight_layout()
plt.show()

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Saved adapter to", OUTPUT_DIR)
print("This directory is gitignored. It is not uploaded.")
wandb.finish()